# Neural Hierarchical Reasoning Transformer (NHRT) Pipeline

This notebook represents the definitive pipeline for the NHRT model. It consolidates:
1. **Adaptive Computation Time (ACT-2)** for continuous latent reasoning.
2. **Low-Rank Adaptation (LoRA)** for parameter-efficient training on a single T4 GPU.
3. **Latent Backtracking with State Perturbation (LBSP)** for evaluating and escaping logical contradictions to improve Exact Accuracy.

Optimized for Kaggle (T4 x2 or single T4).


## 1. Setup Environment

In [ ]:
!git clone https://github.com/jagan25-mj/NHRT.git
%cd NHRT
!pip install setuptools wheel
!pip install einops tqdm pydantic wandb omegaconf hydra-core huggingface_hub peft coolname --quiet


## 2. Generate 6x6 Sudoku Dataset Locally
We generate this locally to avoid hardcoded paths and missing data errors.

In [ ]:
import os, json, random
import numpy as np
from tqdm import tqdm
from dataset.common import PuzzleDatasetMetadata

def is_valid(board, r, c, val):
    if val in board[r]: return False
    if val in board[:, c]: return False
    br, bc = 2 * (r // 2), 3 * (c // 3)
    if val in board[br:br+2, bc:bc+3]: return False
    return True

def solve(board):
    for r in range(6):
        for c in range(6):
            if board[r, c] == 0:
                nums = list(range(1, 7))
                random.shuffle(nums)
                for val in nums:
                    if is_valid(board, r, c, val):
                        board[r, c] = val
                        if solve(board): return True
                        board[r, c] = 0
                return False
    return True

def generate_6x6_board():
    board = np.zeros((6, 6), dtype=int)
    solve(board)
    return board

def generate_puzzles(num_puzzles, num_holes=20):
    inputs, labels = [], []
    for _ in tqdm(range(num_puzzles), desc="Generating 6x6 puzzles"):
        solution = generate_6x6_board()
        puzzle = solution.copy()
        cells = [(r, c) for r in range(6) for c in range(6)]
        random.shuffle(cells)
        for r, c in cells[:num_holes]:
            puzzle[r, c] = 0
        inputs.append(puzzle)
        labels.append(solution)
    return inputs, labels

def convert_subset(set_name, output_dir, num_puzzles, holes=20):
    inputs, labels = generate_puzzles(num_puzzles, holes)
    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)
    for i, (inp, out) in enumerate(zip(inputs, labels)):
        results["inputs"].append(inp.flatten() + 1)
        results["labels"].append(out.flatten() + 1)
        results["puzzle_indices"].append(i + 1)
        results["puzzle_identifiers"].append(0)
        results["group_indices"].append(i + 1)
    results = {k: np.array(v, dtype=np.int32) for k, v in results.items()}
    metadata = PuzzleDatasetMetadata(
        seq_len=36, vocab_size=8, pad_id=0, ignore_label_id=0,
        blank_identifier_id=1, num_puzzle_identifiers=1,
        total_groups=len(results["group_indices"]) - 1,
        mean_puzzle_examples=1, sets=["all"]
    )
    save_dir = os.path.join(output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata.model_dump(), f)
    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)
    with open(os.path.join(output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)

print("Building Sudoku 6x6 Dataset...")
convert_subset("train", "data/sudoku-6x6", 1000)
convert_subset("test", "data/sudoku-6x6", 100)
print("Dataset successfully generated in data/sudoku-6x6!")


## 3. Train NHRT (ACT-2 + LoRA)
We use `finetune_lora.py` with custom paths to ensure it doesn't crash looking for ARC data.

In [ ]:
# Disable WANDB for Kaggle runs unless you have an API key set
import os
os.environ["WANDB_MODE"] = "disabled"

# Define the run command, injecting the base model (HRB) and overriding configs for 6x6
!python finetune_lora.py \
    data_path=data/sudoku-6x6 \
    checkpoint_path=/kaggle/working/checkpoints \
    global_batch_size=64 \
    epochs=400 \
    eval_interval=100 \
    lr=1e-4 \
    lora_rank=8 \
    lora_alpha=16


## 4. Advanced Evaluation: Latent Backtracking (LBSP)
This custom evaluation loop replaces the standard `evaluate_lora.py`. It implements our novel verification mechanism to boost Exact Accuracy by rolling back confident dead-ends in the latent space.

In [ ]:
%%writefile evaluate_backtrack.py
import typing
from typing import List, Dict, Any
import yaml
import os
import copy
import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
import torch.distributed as dist
import pydantic
from omegaconf import OmegaConf

from finetune_lora import PretrainConfig, init_train_state, create_dataloader
from models.losses import IGNORE_LABEL_ID

class EvalConfig(pydantic.BaseModel):
    checkpoint: str
    save_outputs: List[str] = ["inputs", "labels", "puzzle_identifiers", "logits", "q_halt_logits", "q_continue_logits"]
    max_steps: int = 32
    rollback_threshold: float = 1.0 # If max(Q) drops by 1.0, rollback
    noise_std: float = 0.05

def launch():
    eval_cfg = EvalConfig(**OmegaConf.to_container(OmegaConf.from_cli()))
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    RANK = 0
    WORLD_SIZE = 1

    with open(os.path.join(os.path.dirname(eval_cfg.checkpoint), "all_config.yaml"), "r") as f:
        config = PretrainConfig(**yaml.safe_load(f))
        config.eval_save_outputs = eval_cfg.save_outputs
        config.checkpoint_path = "./eval_outputs"
        config.load_checkpoint = None
        config.resume_checkpoint = None
        config.halt_max_steps = eval_cfg.max_steps # override to allow more steps

    eval_loader, eval_metadata = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)

    train_state = init_train_state(config, eval_metadata, world_size=WORLD_SIZE)
    
    try:
        train_state.model.load_state_dict(torch.load(eval_cfg.checkpoint, map_location=device, weights_only=True), assign=True)
    except:
        train_state.model.load_state_dict({k.removeprefix("_orig_mod."): v for k, v in torch.load(eval_cfg.checkpoint, map_location=device, weights_only=True).items()}, assign=True)
    
    print ("Starting Latent Backtracking Evaluation (LBSP)")
    train_state.model.eval()
    
    total_correct = 0
    total_cells = 0
    total_exact = 0
    total_puzzles = 0
    total_rollbacks = 0
    
    with torch.no_grad():
        for set_name, batch, global_batch_size in eval_loader:
            batch = {k: v.cuda().float() if v.is_floating_point() else v.cuda() for k, v in batch.items()}
            
            with torch.device("cuda"):
                carry = train_state.model.initial_carry(batch)
                
            history = []
            
            for step_i in range(eval_cfg.max_steps):
                saved_carry = {
                    "z_H": carry.inner_carry.z_H.clone(),
                    "z_L": carry.inner_carry.z_L.clone(),
                    "steps": carry.steps.clone(),
                    "halted": carry.halted.clone()
                }
                
                carry, loss, metrics, outputs, all_finish = train_state.model(carry=carry, batch=batch, return_keys=["logits", "q_halt_logits", "q_continue_logits"])
                
                q_halt = outputs.get("q_halt_logits", None)
                q_continue = outputs.get("q_continue_logits", None)
                
                rollback_triggered = False
                q_max = None
                if q_halt is not None and q_continue is not None:
                    q_max = torch.maximum(q_halt, q_continue)
                    
                    if len(history) > 0:
                        prev_q_max = history[-1]["q_max"]
                        delta_q = q_max - prev_q_max
                        rollback_mask = (delta_q < -eval_cfg.rollback_threshold) & (~carry.halted)
                        
                        if rollback_mask.any():
                            rollback_triggered = True
                            total_rollbacks += rollback_mask.sum().item()
                            
                            # Apply rollback where mask is true
                            carry.inner_carry.z_H[rollback_mask] = history[-1]["carry"]["z_H"][rollback_mask]
                            carry.inner_carry.z_L[rollback_mask] = history[-1]["carry"]["z_L"][rollback_mask]
                                
                            # Apply perturbation to explore new latent path
                            noise_H = torch.randn_like(carry.inner_carry.z_H[rollback_mask]) * eval_cfg.noise_std
                            noise_L = torch.randn_like(carry.inner_carry.z_L[rollback_mask]) * eval_cfg.noise_std
                            carry.inner_carry.z_H[rollback_mask] += noise_H
                            carry.inner_carry.z_L[rollback_mask] += noise_L
                                
                            carry.steps[rollback_mask] = history[-1]["carry"]["steps"][rollback_mask]
                            carry.halted[rollback_mask] = history[-1]["carry"]["halted"][rollback_mask]
                
                if not rollback_triggered:
                    history.append({"carry": saved_carry, "q_max": q_max})
                
                if all_finish:
                    break
                    
            labels = batch["labels"]
            logits = outputs.get("logits", None)
            
            if logits is not None:
                predictions = torch.argmax(logits, dim=-1)
                mask = labels != IGNORE_LABEL_ID
                correct = (predictions == labels) & mask
                total_correct += correct.sum().item()
                total_cells += mask.sum().item()
                
                for i in range(labels.shape[0]):
                    puzzle_mask = mask[i]
                    if puzzle_mask.sum() > 0:
                        total_puzzles += 1
                        if correct[i].sum() == puzzle_mask.sum():
                            total_exact += 1
                            
    print("\n" + "=" * 60)
    print(f"6x6 SUDOKU LBSP EVALUATION RESULTS")
    print("=" * 60)
    print(f"Cell Accuracy:    {total_correct}/{total_cells} = {100*total_correct/max(total_cells,1):.2f}%")
    print(f"Puzzle Accuracy:  {total_exact}/{total_puzzles} = {100*total_exact/max(total_puzzles,1):.2f}%")
    print(f"Total Rollbacks Triggered: {total_rollbacks}")
    print("=" * 60)

if __name__ == "__main__":
    launch()



In [ ]:
# Find the latest generated LoRA checkpoint automatically
import os, glob
checkpoints = glob.glob("/kaggle/working/checkpoints/**/step_*", recursive=True)
if not checkpoints:
    print("ERROR: No checkpoints found! Did training complete?")
else:
    # Sort by step number
    checkpoints.sort(key=lambda x: int(x.split("_")[-1]) if "_" in x else 0)
    best_ckpt = checkpoints[-1]
    print(f"Found checkpoint: {best_ckpt}")
    
    # Run the LBSP evaluation
    !python evaluate_backtrack.py \
        checkpoint="{best_ckpt}" \
        max_steps=32 \
        rollback_threshold=1.0 \
        noise_std=0.05
